In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
)

from sklearn.multioutput import MultiOutputRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

import joblib
import time
import os
import gc

In [3]:
print("LOADING PHYSICS DATA")

data = np.load("../data/physics_data.npy")

print(f"Dataset shape : {data.shape}")
print(f"Dtype         : {data.dtype}")
print(f"Memory        : {data.nbytes / (1024**3):.2f} GB")

print()

LOADING PHYSICS DATA
Dataset shape : (1000000, 13)
Dtype         : float64
Memory        : 0.10 GB



In [4]:
input_names = [
    "mass",
    "force",
    "initial_velocity",
    "initial_beta",
    "time",
]

target_names = [
    "position",
    "velocity",
    "beta",
    "gamma",
    "momentum",
    "kinetic_energy",
    "total_energy",
    "proper_time",
]

X = data[:, :5]
Y = data[:, 5:13]

print("DATA STRUCTURE")

print("Inputs:")
for i, name in enumerate(input_names):
    print(f"  {i}: {name}")

print()

print("Outputs:")
for i, name in enumerate(target_names):
    print(f"  {i}: {name}")

print()

print("X shape:", X.shape)
print("Y shape:", Y.shape)

DATA STRUCTURE
Inputs:
  0: mass
  1: force
  2: initial_velocity
  3: initial_beta
  4: time

Outputs:
  0: position
  1: velocity
  2: beta
  3: gamma
  4: momentum
  5: kinetic_energy
  6: total_energy
  7: proper_time

X shape: (1000000, 5)
Y shape: (1000000, 8)


In [5]:
print("SPLITTING DATA")

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42,
)

print(f"Training   : {X_train.shape[0]:,}")
print(f"Validation : {X_val.shape[0]:,}")
print(f"Test       : {X_test.shape[0]:,}")

print()

SPLITTING DATA
Training   : 800,000
Validation : 100,000
Test       : 100,000



In [6]:
print("SCALING DATA")

X_scaler = StandardScaler()
Y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_val_scaled = X_scaler.transform(X_val)
X_test_scaled = X_scaler.transform(X_test)

Y_train_scaled = Y_scaler.fit_transform(Y_train)
Y_val_scaled = Y_scaler.transform(Y_val)
Y_test_scaled = Y_scaler.transform(Y_test)

print("Scaling complete.")

# Save scalers immediately
os.makedirs("models_physics_data", exist_ok=True)

joblib.dump(
    X_scaler,
    "models_physics_data/X_scaler.joblib",
)

joblib.dump(
    Y_scaler,
    "models_physics_data/Y_scaler.joblib",
)

print("Scalers saved.")

print()

SCALING DATA
Scaling complete.
Scalers saved.



In [8]:
def evaluate_model(
    model_name,
    model,
    X_eval,
    Y_eval,
    y_scaler=None,
):
    print()
    print(f"EVALUATING: {model_name}")

    start = time.perf_counter()

    predictions = model.predict(X_eval)

    prediction_time = time.perf_counter() - start

    # Convert predictions back to physical units
    if y_scaler is not None:
        predictions = y_scaler.inverse_transform(
            predictions
        )

    rows = []

    for i, target in enumerate(target_names):

        rmse = np.sqrt(
            mean_squared_error(
                Y_eval[:, i],
                predictions[:, i],
            )
        )

        mae = mean_absolute_error(
            Y_eval[:, i],
            predictions[:, i],
        )

        r2 = r2_score(
            Y_eval[:, i],
            predictions[:, i],
        )

        rows.append({
            "model": model_name,
            "target": target,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
        })

        print(
            f"{target:18s} "
            f"R²={r2:12.6f} "
            f"RMSE={rmse:.6e}"
        )

    print()
    print(
        f"Prediction time: "
        f"{prediction_time:.6f} sec"
    )

    # Explicitly free predictions
    del predictions
    gc.collect()

    return pd.DataFrame(rows)

In [9]:
print("PREPARING RESULT STORAGE")

results_path = "physics_data_sklearn_results.csv"

all_results = []

# Remove old result file if it exists
if os.path.exists(results_path):
    os.remove(results_path)

print(f"Results will be saved to: {results_path}")
print()

PREPARING RESULT STORAGE
Results will be saved to: physics_data_sklearn_results.csv



In [10]:
print()
print("MODEL 1/7 — LINEAR REGRESSION")

linear_model = LinearRegression()

start = time.perf_counter()

linear_model.fit(
    X_train_scaled,
    Y_train_scaled,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
linear_results = evaluate_model(
    "Linear Regression",
    linear_model,
    X_test_scaled,
    Y_test,
    Y_scaler,
)

# Add to results
all_results.append(linear_results)

# Save model IMMEDIATELY
linear_path = (
    "models_physics_data/"
    "linear_regression.joblib"
)

joblib.dump(
    linear_model,
    linear_path,
)

print(f"Model saved to: {linear_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del linear_model
del linear_results

gc.collect()

print("Linear Regression model deleted from RAM.")


MODEL 1/7 — LINEAR REGRESSION
Training time: 0.219 sec

EVALUATING: Linear Regression
position           R²=    0.303911 RMSE=2.822741e+12
velocity           R²=    0.499636 RMSE=9.117771e+07
beta               R²=    0.499636 RMSE=3.041361e-01
gamma              R²=    0.011573 RMSE=1.341583e+04
momentum           R²=    0.117497 RMSE=2.156966e+14
kinetic_energy     R²=    0.059168 RMSE=6.323254e+22
total_energy       R²=    0.190966 RMSE=6.323254e+22
proper_time        R²=    0.680157 RMSE=5.504368e+03

Prediction time: 0.004631 sec
Model saved to: models_physics_data/linear_regression.joblib
Results saved to: physics_data_sklearn_results.csv
Linear Regression model deleted from RAM.


In [11]:
print()
print("MODEL 2/7 — POLYNOMIAL REGRESSION DEGREE 2")

poly2_model = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=2,
            include_bias=False,
        ),
    ),
    (
        "regressor",
        LinearRegression(),
    ),
])

start = time.perf_counter()

poly2_model.fit(
    X_train_scaled,
    Y_train_scaled,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
poly2_results = evaluate_model(
    "Polynomial Degree 2",
    poly2_model,
    X_test_scaled,
    Y_test,
    Y_scaler,
)

all_results.append(poly2_results)

# Save immediately
poly2_path = (
    "models_physics_data/"
    "polynomial_degree_2.joblib"
)

joblib.dump(
    poly2_model,
    poly2_path,
)

print(f"Model saved to: {poly2_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del poly2_model
del poly2_results

gc.collect()

print("Polynomial Degree 2 model deleted from RAM.")


MODEL 2/7 — POLYNOMIAL REGRESSION DEGREE 2
Training time: 0.684 sec

EVALUATING: Polynomial Degree 2
position           R²=    0.510478 RMSE=2.367145e+12
velocity           R²=    0.533242 RMSE=8.806258e+07
beta               R²=    0.533242 RMSE=2.937452e-01
gamma              R²=    0.021429 RMSE=1.334877e+04
momentum           R²=    0.227554 RMSE=2.017991e+14
kinetic_energy     R²=    0.134703 RMSE=6.064111e+22
total_energy       R²=    0.255920 RMSE=6.064111e+22
proper_time        R²=    0.758253 RMSE=4.785417e+03

Prediction time: 0.023439 sec
Model saved to: models_physics_data/polynomial_degree_2.joblib
Results saved to: physics_data_sklearn_results.csv
Polynomial Degree 2 model deleted from RAM.


In [12]:
print()
print("MODEL 3/7 — DECISION TREE")

tree_model = DecisionTreeRegressor(
    max_depth=25,
    min_samples_leaf=10,
    random_state=42,
)

start = time.perf_counter()

tree_model.fit(
    X_train,
    Y_train,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
tree_results = evaluate_model(
    "Decision Tree",
    tree_model,
    X_test,
    Y_test,
)

all_results.append(tree_results)

# Save immediately
tree_path = (
    "models_physics_data/"
    "decision_tree.joblib"
)

joblib.dump(
    tree_model,
    tree_path,
)

print(f"Model saved to: {tree_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del tree_model
del tree_results

gc.collect()

print("Decision Tree model deleted from RAM.")


MODEL 3/7 — DECISION TREE
Training time: 15.237 sec

EVALUATING: Decision Tree
position           R²=    0.694438 RMSE=1.870202e+12
velocity           R²=    0.742821 RMSE=6.536776e+07
beta               R²=    0.742821 RMSE=2.180434e-01
gamma              R²=    0.136381 RMSE=1.254025e+04
momentum           R²=    0.803052 RMSE=1.018970e+14
kinetic_energy     R²=    0.781484 RMSE=3.047376e+22
total_energy       R²=    0.811864 RMSE=3.049250e+22
proper_time        R²=    0.716176 RMSE=5.185174e+03

Prediction time: 0.017651 sec
Model saved to: models_physics_data/decision_tree.joblib
Results saved to: physics_data_sklearn_results.csv
Decision Tree model deleted from RAM.


In [13]:
print()
print("MODEL 4/7 — GRADIENT BOOSTING")

gb_base = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
)

gb_model = MultiOutputRegressor(
    gb_base,
    n_jobs=-1,
)

start = time.perf_counter()

gb_model.fit(
    X_train,
    Y_train,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
gb_results = evaluate_model(
    "Gradient Boosting",
    gb_model,
    X_test,
    Y_test,
)

all_results.append(gb_results)

# Save immediately
gb_path = (
    "models_physics_data/"
    "gradient_boosting.joblib"
)

joblib.dump(
    gb_model,
    gb_path,
)

print(f"Model saved to: {gb_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del gb_base
del gb_model
del gb_results

gc.collect()

print("Gradient Boosting model deleted from RAM.")


MODEL 4/7 — GRADIENT BOOSTING
Training time: 1037.024 sec

EVALUATING: Gradient Boosting
position           R²=    0.975087 RMSE=5.340144e+11
velocity           R²=    0.933747 RMSE=3.317785e+07
beta               R²=    0.933747 RMSE=1.106694e-01
gamma              R²=    0.952659 RMSE=2.936060e+03
momentum           R²=    0.789636 RMSE=1.053102e+14
kinetic_energy     R²=    0.762804 RMSE=3.174960e+22
total_energy       R²=    0.823176 RMSE=2.956156e+22
proper_time        R²=    0.984418 RMSE=1.214939e+03

Prediction time: 0.692446 sec
Model saved to: models_physics_data/gradient_boosting.joblib
Results saved to: physics_data_sklearn_results.csv
Gradient Boosting model deleted from RAM.


In [14]:
print()
print("MODEL 5/7 — HIST GRADIENT BOOSTING")

hist_base = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1e-3,
    random_state=42,
)

hist_model = MultiOutputRegressor(
    hist_base,
    n_jobs=-1,
)

start = time.perf_counter()

hist_model.fit(
    X_train,
    Y_train,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
hist_results = evaluate_model(
    "Hist Gradient Boosting",
    hist_model,
    X_test,
    Y_test,
)

all_results.append(hist_results)

# Save immediately
hist_path = (
    "models_physics_data/"
    "hist_gradient_boosting.joblib"
)

joblib.dump(
    hist_model,
    hist_path,
)

print(f"Model saved to: {hist_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del hist_base
del hist_model
del hist_results

gc.collect()

print("Hist Gradient Boosting model deleted from RAM.")


MODEL 5/7 — HIST GRADIENT BOOSTING
Training time: 12.151 sec

EVALUATING: Hist Gradient Boosting
position           R²=    0.989759 RMSE=3.423848e+11
velocity           R²=    0.965993 RMSE=2.377013e+07
beta               R²=    0.965993 RMSE=7.928862e-02
gamma              R²=    0.980316 RMSE=1.893208e+03
momentum           R²=    0.483172 RMSE=1.650662e+14
kinetic_energy     R²=    0.428487 RMSE=4.928308e+22
total_energy       R²=    0.513071 RMSE=4.905576e+22
proper_time        R²=    0.993153 RMSE=8.053736e+02

Prediction time: 0.982607 sec
Model saved to: models_physics_data/hist_gradient_boosting.joblib
Results saved to: physics_data_sklearn_results.csv
Hist Gradient Boosting model deleted from RAM.


In [15]:
print()
print("MODEL 6/7 — CONTROLLED RANDOM FOREST")

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=10,
    max_features=1.0,
    n_jobs=-1,
    random_state=42,
)

start = time.perf_counter()

rf_model.fit(
    X_train,
    Y_train,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
rf_results = evaluate_model(
    "Random Forest Controlled",
    rf_model,
    X_test,
    Y_test,
)

all_results.append(rf_results)

# Save immediately
rf_path = (
    "models_physics_data/"
    "random_forest_controlled.joblib"
)

joblib.dump(
    rf_model,
    rf_path,
)

print(f"Model saved to: {rf_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del rf_model
del rf_results

gc.collect()

print("Random Forest model deleted from RAM.")


MODEL 6/7 — CONTROLLED RANDOM FOREST
Training time: 124.077 sec

EVALUATING: Random Forest Controlled
position           R²=    0.678221 RMSE=1.919189e+12
velocity           R²=    0.628802 RMSE=7.853227e+07
beta               R²=    0.628802 RMSE=2.619555e-01
gamma              R²=    0.180366 RMSE=1.221673e+04
momentum           R²=    0.795919 RMSE=1.037257e+14
kinetic_energy     R²=    0.772891 RMSE=3.106719e+22
total_energy       R²=    0.804920 RMSE=3.105011e+22
proper_time        R²=    0.742660 RMSE=4.937336e+03

Prediction time: 0.240803 sec
Model saved to: models_physics_data/random_forest_controlled.joblib
Results saved to: physics_data_sklearn_results.csv
Random Forest model deleted from RAM.


In [16]:
print()
print("MODEL 7/7 — CONTROLLED EXTRA TREES")

extra_model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=10,
    max_features=1.0,
    n_jobs=-1,
    random_state=42,
)

start = time.perf_counter()

extra_model.fit(
    X_train,
    Y_train,
)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.3f} sec")

# Evaluate
extra_results = evaluate_model(
    "Extra Trees Controlled",
    extra_model,
    X_test,
    Y_test,
)

all_results.append(extra_results)

# Save immediately
extra_path = (
    "models_physics_data/"
    "extra_trees_controlled.joblib"
)

joblib.dump(
    extra_model,
    extra_path,
)

print(f"Model saved to: {extra_path}")

# Save results immediately
pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    results_path,
    index=False,
)

print(f"Results saved to: {results_path}")

# Free RAM
del extra_model
del extra_results

gc.collect()

print("Extra Trees model deleted from RAM.")


MODEL 7/7 — CONTROLLED EXTRA TREES
Training time: 40.555 sec

EVALUATING: Extra Trees Controlled
position           R²=    0.695578 RMSE=1.866708e+12
velocity           R²=    0.710342 RMSE=6.937270e+07
beta               R²=    0.710342 RMSE=2.314024e-01
gamma              R²=    0.185201 RMSE=1.218065e+04
momentum           R²=    0.710132 RMSE=1.236190e+14
kinetic_energy     R²=    0.677429 RMSE=3.702522e+22
total_energy       R²=    0.722716 RMSE=3.701855e+22
proper_time        R²=    0.802568 RMSE=4.324617e+03

Prediction time: 0.262938 sec
Model saved to: models_physics_data/extra_trees_controlled.joblib
Results saved to: physics_data_sklearn_results.csv
Extra Trees model deleted from RAM.


In [17]:
print()
print("COMBINING FINAL RESULTS")

all_results_df = pd.concat(
    all_results,
    ignore_index=True,
)

print()

print(
    all_results_df[
        ["model", "target", "R2"]
    ].to_string(index=False)
)

# Save final results
all_results_df.to_csv(
    results_path,
    index=False,
)

print()
print(f"Final results saved to: {results_path}")


COMBINING FINAL RESULTS

                   model         target       R2
       Linear Regression       position 0.303911
       Linear Regression       velocity 0.499636
       Linear Regression           beta 0.499636
       Linear Regression          gamma 0.011573
       Linear Regression       momentum 0.117497
       Linear Regression kinetic_energy 0.059168
       Linear Regression   total_energy 0.190966
       Linear Regression    proper_time 0.680157
     Polynomial Degree 2       position 0.510478
     Polynomial Degree 2       velocity 0.533242
     Polynomial Degree 2           beta 0.533242
     Polynomial Degree 2          gamma 0.021429
     Polynomial Degree 2       momentum 0.227554
     Polynomial Degree 2 kinetic_energy 0.134703
     Polynomial Degree 2   total_energy 0.255920
     Polynomial Degree 2    proper_time 0.758253
           Decision Tree       position 0.694438
           Decision Tree       velocity 0.742821
           Decision Tree           beta 0.7

In [18]:
print()
print("BEST MODEL FOR EACH TARGET")

best_models = (
    all_results_df
    .sort_values(
        "R2",
        ascending=False,
    )
    .groupby("target")
    .first()
    .reset_index()
)

print()

print(
    best_models[
        ["target", "model", "R2"]
    ].to_string(index=False)
)


BEST MODEL FOR EACH TARGET

        target                  model       R2
          beta Hist Gradient Boosting 0.965993
         gamma Hist Gradient Boosting 0.980316
kinetic_energy          Decision Tree 0.781484
      momentum          Decision Tree 0.803052
      position Hist Gradient Boosting 0.989759
   proper_time Hist Gradient Boosting 0.993153
  total_energy      Gradient Boosting 0.823176
      velocity Hist Gradient Boosting 0.965993


In [19]:
print()
print("OVERALL MODEL RANKING")

model_ranking = (
    all_results_df
    .groupby("model")["R2"]
    .mean()
    .sort_values(
        ascending=False,
    )
)

print()

print(model_ranking)


OVERALL MODEL RANKING

model
Gradient Boosting           0.894409
Hist Gradient Boosting      0.789993
Decision Tree               0.678630
Random Forest Controlled    0.654073
Extra Trees Controlled      0.651788
Polynomial Degree 2         0.371853
Linear Regression           0.295318
Name: R2, dtype: float64


In [20]:
print()
print("MODEL FILE SIZES")

model_directory = "models_physics_data"

for filename in sorted(
    os.listdir(model_directory)
):

    path = os.path.join(
        model_directory,
        filename,
    )

    if os.path.isfile(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            f"{filename:45s} "
            f"{size_mb:10.2f} MB"
        )


MODEL FILE SIZES
X_scaler.joblib                                     0.00 MB
Y_scaler.joblib                                     0.00 MB
decision_tree.joblib                                7.27 MB
extra_trees_controlled.joblib                     337.31 MB
gradient_boosting.joblib                            5.44 MB
hist_gradient_boosting.joblib                       4.76 MB
linear_regression.joblib                            0.00 MB
polynomial_degree_2.joblib                          0.00 MB
random_forest_controlled.joblib                   293.26 MB


In [21]:
print()
print("FINAL MEMORY CLEANUP")

del X_train
del X_temp
del X_test

del Y_train
del Y_temp
del Y_test

del X_train_scaled
del X_val_scaled
del X_test_scaled

del Y_train_scaled
del Y_val_scaled
del Y_test_scaled

del X
del Y

del data

del X_scaler
del Y_scaler

del all_results
del all_results_df
del best_models
del model_ranking

gc.collect()

print("Large training arrays and remaining objects deleted.")
print("Garbage collection completed.")


FINAL MEMORY CLEANUP
Large training arrays and remaining objects deleted.
Garbage collection completed.
